# Solution C1 · Features in euros sink the KNN: where the distance goes

**Type**: Break the model

## Objective

Build a dataset in which twenty irrelevant features, expressed in large units, alone decide the distances between the points, and
measure by how much the KNN accuracy collapses and how much of the distance each group of features explains, with and without scaling.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_moons
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
X_info, y = make_moons(n_samples=600, noise=0.25, random_state=0)      # two informative features, scale about 1
junk = rng.normal(size=(600, 20)) * 1000                               # twenty irrelevant features, in "euros"
X = np.column_stack([X_info, junk])
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Commented solution

In [ ]:
acc_raw = cross_val_score(KNeighborsClassifier(5), X, y, cv=cv).mean()
acc_scaled = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(5)), X, y, cv=cv).mean()
acc_without_junk = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(5)), X[:, :2], y, cv=cv).mean()
print(f"raw {acc_raw:.3f} | scaled {acc_scaled:.3f} | without the useless features {acc_without_junk:.3f}")

In [ ]:
def distance_share(X, n_pairs=3000, seed=0):
    r = np.random.default_rng(seed)
    i, j = r.integers(0, len(X), n_pairs), r.integers(0, len(X), n_pairs)
    d2 = (X[i] - X[j]) ** 2                      # contribution of every column to the squared distance
    return d2.sum(axis=0) / d2.sum()


raw_share = distance_share(X)
scaled_share = distance_share(StandardScaler().fit_transform(X))
info_raw, info_scaled = raw_share[:2].sum(), scaled_share[:2].sum()
print(f"share of the distance of the 2 informative columns: raw {info_raw:.2e} | scaled {info_scaled:.3f}")

## Check

In [ ]:
with verify():
    check(acc_raw < 0.62, "without scaling the KNN is close to chance (accuracy below 0.62)", "with the feature in 'euros' the KNN should collapse")
    check(acc_scaled > acc_raw + 0.05, "scaling recovers part of the loss", "acc_scaled must exceed acc_raw")
    check(acc_without_junk > acc_scaled + 0.15,
          "but removing the useless features helps much more: scaling does not fix relevance",
          "without the junk columns the accuracy must clearly exceed the scaled one")
    check(len(raw_share) == 22 and abs(raw_share.sum() - 1) < 1e-9, "the 22 shares sum to 1", "distance_share returns one fraction per column, and the sum is 1")
    check(info_raw < 1e-5, "without scaling the two informative columns explain less than 0.001% of the distance", "info_raw = raw_share[:2].sum()")
    check(0.05 < info_scaled < 0.2, "after scaling the informative columns weigh about 2/22, that is less than 20%",
          "after scaling every column weighs about 1/22: the two informative ones about 9%")
done()

## What you learned

- Without scaling the twenty `junk` columns explain practically 100% of the distance: the "neighbors" of a point are the points with similar
  `junk`, that is random ones. The KNN drops to about 0.5, that is chance level.
- With scaling every column weighs about one twenty-second and the accuracy goes back up, but it does **not** reach the one obtained by removing
  `junk`: the twenty useless columns keep pushing the neighbors at random for about 90% of the distance.
- Scaling solves a problem of **units**, not of **relevance**. For the KNN you also need feature selection or per-feature
  weights: it is one of the reasons why the KNN suffers in high dimensions.

## Common pitfall

Scaling and considering the job done. Irrelevant features have the same weight as relevant ones in a Euclidean
distance; in a real dataset with dozens of columns, most of them useless, the scaled KNN can still
do badly.

## Interview question

<details><summary>I standardized the features but the KNN still does badly. What do I check?</summary>

That the irrelevant features are not diluting the distance: with many barely informative columns the neighbors become almost
equidistant (curse of dimensionality). I try feature selection, a dimensionality reduction (PCA),
the metric (Manhattan is less sensitive) or weighting the features. And I check that k is chosen in cross-validation.

</details>